# Train-Test Split, Stratification, and Data Leakage
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

Train-test split mechanics, split ratios, random_state, stratify, and a deliberate data leakage demonstration.

### Main goals:

- Show why evaluating on training data is misleading.
- Build a working data leakage example by fitting a scaler incorrectly.
- Compare different split ratios.
- Confirm what random_state and stratify actually control.

---

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

## Why Split at All

In [ ]:
np.random.seed(10)
n = 300
X = np.random.randn(n, 5)
true_weights = np.array([1.2, -0.8, 0.5, 0.0, 0.0])
y_prob = 1 / (1 + np.exp(-(X @ true_weights)))
y = (y_prob > 0.5).astype(int)

model = LogisticRegression().fit(X, y)
same_data_acc = accuracy_score(y, model.predict(X))
print(f'Accuracy on training data itself: {same_data_acc:.4f}')

**Observation:**
Evaluating on the same data used for training tells you almost nothing about real performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model2 = LogisticRegression().fit(X_train, y_train)

train_acc = accuracy_score(y_train, model2.predict(X_train))
test_acc = accuracy_score(y_test, model2.predict(X_test))
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'Train acc: {train_acc:.4f}  Test acc: {test_acc:.4f}')

**Observation:**
Test accuracy is lower than the same-data accuracy above — a small drop here, but the trustworthy number, since the model never saw these rows.

## Data Leakage

In [ ]:
np.random.seed(3)
n2 = 200
X_leak = np.random.randn(n2, 4)
y_leak = (X_leak[:, 0] + X_leak[:, 1] > 0).astype(int)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y_leak, test_size=0.2, random_state=1)

# WRONG: scaler fit on full dataset (train+test combined)
scaler_wrong = StandardScaler().fit(X_leak)
X_train_wrong = scaler_wrong.transform(X_train_l)
X_test_wrong = scaler_wrong.transform(X_test_l)

# RIGHT: scaler fit on training data only
scaler_right = StandardScaler().fit(X_train_l)
X_train_right = scaler_right.transform(X_train_l)
X_test_right = scaler_right.transform(X_test_l)

model_wrong = LogisticRegression().fit(X_train_wrong, y_train_l)
model_right = LogisticRegression().fit(X_train_right, y_train_l)

print(f'Leaked (scaler fit on all data):     {accuracy_score(y_test_l, model_wrong.predict(X_test_wrong)):.4f}')
print(f'Correct (scaler fit on train only):  {accuracy_score(y_test_l, model_right.predict(X_test_right)):.4f}')

**Observation:**
The difference between the leaked and correct pipeline is small, a couple of points, not a dramatic jump. Leakage does not always announce itself with an obviously wrong result.

In [ ]:
print('Scaler mean (full data):    ', scaler_wrong.mean_.round(3))
print('Scaler mean (train only):   ', scaler_right.mean_.round(3))

**Observation:**
The scaler fit on the full dataset has already used the test set's statistics before any prediction is made, even though the model itself never sees the test labels directly.

## Split Ratios

In [ ]:
results = []
for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=ratio, random_state=42)
    m = LogisticRegression().fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    results.append({'test_size': ratio, 'n_train': len(Xtr), 'n_test': len(Xte), 'test_accuracy': acc})

results_df = pd.DataFrame(results)
results_df

**Observation:**
Accuracy does not move much across ratios, but test_size=0.1 leaves only 30 test samples, so one misclassification swings the result by 3%+. 80/20 is a reasonable default rather than a mathematically special choice.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(results_df['test_size'], results_df['test_accuracy'], 'o-', color='#1F3864', lw=2, markersize=8)
ax.set_xlabel('test_size')
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy Across Split Ratios')
ax.set_ylim(0.85, 1.0)
plt.tight_layout()
plt.show()

## random_state

In [ ]:
Xtr1, Xte1, ytr1, yte1 = train_test_split(X, y, test_size=0.2, random_state=42)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X, y, test_size=0.2, random_state=42)
print('Same random_state called twice — identical split?', np.array_equal(Xte1, Xte2))

**Observation:**
The same random_state produces an identical split every time. Matters when comparing two models on the same data.

## stratify

In [ ]:
np.random.seed(5)
n3 = 200
X_imb = np.random.randn(n3, 3)
y_imb = np.random.choice([0, 1], size=n3, p=[0.9, 0.1])

print('Overall:', pd.Series(y_imb).value_counts().to_dict())

_, _, _, yte_no = train_test_split(X_imb, y_imb, test_size=0.2, random_state=7)
print('No stratify  - test distribution:', pd.Series(yte_no).value_counts().to_dict())

_, _, _, yte_strat = train_test_split(X_imb, y_imb, test_size=0.2, random_state=7, stratify=y_imb)
print('Stratified   - test distribution:', pd.Series(yte_strat).value_counts().to_dict())

**Observation:**
Difference looks small on this run, but without stratify the minority class can drop to 1–2 samples in the test set purely by chance. stratify keeps the class ratio consistent across the split.